# Preprocessing Technique: Outlier Detection (Review Length)
### IT2011 — Progress Review I: Data Preprocessing and EDA
**Presented by:** Member 4 — *[Full Name, IT Number]*
**Assigned dataset:** Rotten Tomatoes Movie Review Dataset (Cornell)

This notebook covers my individually-owned preprocessing technique for our group's project,
as required for Progress Review I: technique explanation, justification, implementation, and
an interpreted EDA visualization.


## Shared Setup

This cell is identical across every member's notebook so each person's notebook can run
independently. It loads the assigned dataset and converts it to a pandas DataFrame.


In [ ]:
!pip install -q datasets scikit-learn pandas matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset

sns.set_style("whitegrid")

ds = load_dataset("cornell-movie-review-data/rotten_tomatoes")
train_df = ds["train"].to_pandas()
val_df = ds["validation"].to_pandas()
test_df = ds["test"].to_pandas()

print("Train:", train_df.shape, "| Validation:", val_df.shape, "| Test:", test_df.shape)
train_df.head()


## 1. Technique Explanation

**Outlier detection/removal** identifies data points that are numerically extreme compared to
the rest of the dataset. Since our raw feature is text rather than a number, we derive a
numeric feature — **review length in words** — and apply the standard IQR (interquartile
range) method to flag unusually short or unusually long reviews.

## 2. Justification for This Dataset

Extremely short reviews (e.g., a single word) may carry too little signal to classify reliably,
and extremely long reviews may be data artifacts (e.g., accidentally concatenated entries)
rather than typical single-sentence reviews. Identifying these cases lets the group make an
informed, documented decision about whether to keep, flag, or exclude them — rather than
leaving it unexamined.


## 3. Implementation

In [ ]:
train_df["word_count"] = train_df["text"].str.split().apply(len)

Q1 = train_df["word_count"].quantile(0.25)
Q3 = train_df["word_count"].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = train_df[(train_df["word_count"] < lower_bound) | (train_df["word_count"] > upper_bound)]

print(f"Q1={Q1}, Q3={Q3}, IQR={IQR}")
print(f"Outlier bounds: [{lower_bound:.1f}, {upper_bound:.1f}] words")
print(f"Outliers found: {len(outliers)} out of {len(train_df)} ({len(outliers)/len(train_df):.1%})")
outliers[["text", "word_count"]].sort_values("word_count", ascending=False).head(5)


## 4. EDA Visualization & Interpretation

A boxplot visualizes the distribution of review length and highlights outliers directly.


In [ ]:
plt.figure(figsize=(7, 4))
sns.boxplot(data=train_df, x="label", y="word_count", palette={0: "#C1272D", 1: "#2E9E6D"})
plt.xticks([0, 1], ["Negative", "Positive"])
plt.title("Review Length Distribution by Sentiment (Outlier Detection)")
plt.xlabel("")
plt.ylabel("Word count")
plt.show()


**Interpretation:** [Fill in after running — describe: how many outliers were found,
whether they skew toward one sentiment class, and whether the group decided to keep them
(recommended default: keep them, since single-sentence movie reviews naturally vary in length
and removing them risks discarding valid data — but document that this was a deliberate
decision, not an oversight).]
